# STEP 2. QA 데이터셋 자동 생성
EXAONE으로 지식베이스 청크에서 QA 쌍 자동 생성 → JSON 저장

In [1]:
# [셀1] 경로 설정 + FAISS 인덱스 로드
from pathlib import Path
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

INDEX_DIR = Path(r"D:\rag_index")
QA_PATH   = Path(r"D:\충북대\지능화캡스톤\data\qa_dataset.json")

print("임베딩 모델 로딩 중...")
embed_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
vectordb = FAISS.load_local(
    str(INDEX_DIR), embed_model,
    allow_dangerous_deserialization=True
)
print(f"인덱스 로드 완료: {vectordb.index.ntotal}개 벡터")

임베딩 모델 로딩 중...


C:\Users\hyebi\AppData\Local\Temp\ipykernel_53412\49915418.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embed_model = HuggingFaceEmbeddings(
d:\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 38441.07it/s]


인덱스 로드 완료: 305개 벡터


In [2]:
# [셀2] 재난 유형별 대표 청크 수집
# 각 유형에서 청크를 샘플링해 QA 생성 원천 데이터로 사용
from collections import defaultdict
import random

# docstore에서 전체 청크 꺼내기
all_chunks = list(vectordb.docstore._dict.values())
print(f"전체 청크: {len(all_chunks)}개")

# 유형별로 묶기
by_type = defaultdict(list)
for chunk in all_chunks:
    dtype = chunk.metadata.get("disaster_type", "기타")
    by_type[dtype].append(chunk)

print("\n유형별 청크 수:")
for dtype, chunks in sorted(by_type.items(), key=lambda x: -len(x[1])):
    print(f"  {dtype}: {len(chunks)}개")

# 유형별 목표 QA 수 설정 (총 60개 목표)
# 청크가 많을수록 QA도 많이 생성
QA_TARGET = {
    "지진":        12,
    "화재":        12,
    "황사":         8,
    "화학사고재난":  8,
    "미세먼지":     6,   # 2→24로 늘었으니 상향
    "감염병예방":   6,
    "강풍":         5,   # 3→10으로 늘었으니 상향
    "산불":         5,   # 2→10으로 늘었으니 추가
    "한파":         4,
    "대설":         4,
}
print(f"\n목표 QA 총합: {sum(QA_TARGET.values())}개")

전체 청크: 305개

유형별 청크 수:
  지진: 92개
  황사: 61개
  화재: 53개
  화학사고재난: 27개
  미세먼지: 24개
  감염병예방: 12개
  강풍: 10개
  산불: 10개
  한파: 5개
  대설: 4개
  태풍: 2개
  호우: 2개
  홍수: 2개
  풍랑: 1개

목표 QA 총합: 70개


In [3]:
# [셀3] EXAONE LLM 연결
from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model="exaone3.5:7.8b",
    temperature=0.3,   # QA 생성은 약간의 다양성 허용
    num_ctx=4096,
)

# 연결 테스트
test = llm.invoke("안녕하세요. 한 줄로 답하세요.")
print("LLM 연결 OK:", test[:80])

LLM 연결 OK: 안녕하세요! 어떻게 도와드릴까요? 😊


In [9]:
import json, re

QA_PROMPT = """당신은 재난안전 전문가입니다.
아래 [문서 내용]을 읽고 QA를 {n}개 만드세요.

규칙:
1. gold_answer는 반드시 [문서 내용]에 있는 구체적인 내용이어야 합니다.
2. "이렇게 합니다", "확인하세요" 같은 빈 답변은 절대 금지입니다.
3. 질문 유형: 절차형 / 수치법령형 / 안전판단형 / 복합추론형
4. JSON 배열만 출력하세요.

[문서 내용]
{context}

[출력 형식]
[
  {{"type": "절차형", "question": "질문", "gold_answer": "구체적인 정답"}},
  ...
]"""


def get_rich_context(dtype: str, n_chunks: int = 5) -> str:
    """같은 유형의 청크 여러 개를 합쳐 풍부한 컨텍스트 생성"""
    chunks = by_type.get(dtype, [])
    selected = chunks[:n_chunks]
    combined = "\n\n".join(c.page_content for c in selected)
    return combined[:2000]  # 800→2000자로 확대


def generate_qa(context: str, n: int = 2) -> list:
    prompt = QA_PROMPT.format(context=context, n=n)
    response = llm.invoke(prompt)
    try:
        cleaned = re.sub(r"```json|```", "", response).strip()
        match = re.search(r"\[.*\]", cleaned, re.DOTALL)
        if match:
            qas = json.loads(match.group())
            # 빈 답변 필터링
            return [q for q in qas
                    if q.get("gold_answer") and
                    len(q["gold_answer"]) > 20 and
                    "이렇게" not in q["gold_answer"][:10]]
    except Exception as e:
        print(f"  [WARN] 파싱 실패: {e}")
    return []


# 테스트
ctx = get_rich_context("지진", n_chunks=5)
print(f"컨텍스트 길이: {len(ctx)}자")
print(f"컨텍스트 미리보기:\n{ctx[:300]}\n")

test_qa = generate_qa(ctx, n=2)
print("테스트 QA:")
for q in test_qa:
    print(f"  [{q.get('type')}] {q.get('question')}")
    print(f"  → {q.get('gold_answer')}")

컨텍스트 길이: 1917자
컨텍스트 미리보기:
지진, 평소에 이렇게 대비합니다. 102
지진 국민행동요령
미리 대비하고 
알아두는 지진이야기
지진, 평소에 이렇게 대비합니다.	
01
 
지진이 발생하면 이렇게 대피합니다.	
03
 
장소에 따라 이렇게 행동합니다.	
07
 
몸이 불편하신 분은 이렇게 행동합니다.	
11
 
보호자(조력자)는 이렇게 행동합니다.	
13 
 
어린이와 함께 있을 때에는 이렇게 행동합니다.	
15
 
흔들림이 멈추거나 대피 후에는 이렇게 행동합니다.	
16 
 
지진해일이 발생하면 이렇게 대피합니다.	
20
 
해외에서 지진 발생 시 이렇게 대처합니

테스트 QA:
  [절차형] 집 안에서 지진 발생 시 안전한 대피 공간으로 적합한 장소는 어디인가요?
  → 탁자 아래와 같이 집 안에서 대피할 수 있는 안전한 대피 공간을 미리 파악해 두는 것이 좋습니다.
  [안전판단형] 지진 발생 시 깨진 유리로부터 안전을 확보하기 위해 어떤 조치를 취해야 하나요?
  → 두꺼운 실내화를 준비해 두어 깨진 유리 등에 다치지 않도록 합니다.


In [10]:
# [셀5] 전체 QA 자동 생성 (시간 소요: 유형·수량에 따라 10~30분)
from tqdm import tqdm
import time

qa_dataset = []
qa_id = 1

for dtype, target_n in QA_TARGET.items():
    if not by_type.get(dtype):
        print(f"[SKIP] {dtype}")
        continue

    print(f"\n[{dtype}] {target_n}개 생성 중...")
    collected = []

    # 청크를 5개씩 묶어서 컨텍스트 생성
    chunks = by_type[dtype]
    pbar = tqdm(total=target_n, desc=dtype)
    
    i = 0
    while len(collected) < target_n and i < len(chunks):
        ctx = "\n\n".join(c.page_content for c in chunks[i:i+5])[:2000]
        n_req = min(2, target_n - len(collected))
        qas = generate_qa(ctx, n=n_req)
        for qa in qas:
            collected.append({
                "id": f"Q{qa_id:03d}",
                "type": qa.get("type", "절차형"),
                "disaster_type": dtype,
                "question": qa["question"],
                "gold_answer": qa["gold_answer"],
                "source_file": chunks[i].metadata.get("source_file", ""),
            })
            qa_id += 1
            pbar.update(1)
            if len(collected) >= target_n:
                break
        i += 5
        time.sleep(0.3)

    pbar.close()
    qa_dataset.extend(collected)
    print(f"  → {len(collected)}개 완료")

print(f"\n===== 총 {len(qa_dataset)}개 QA 생성 완료 =====")


[지진] 12개 생성 중...


지진: 100%|██████████| 12/12 [00:32<00:00,  2.69s/it]


  → 12개 완료

[화재] 12개 생성 중...


화재: 100%|██████████| 12/12 [00:37<00:00,  3.09s/it]


  → 12개 완료

[황사] 8개 생성 중...


...

  → 4개 완료

[한파] 4개 생성 중...


한파:  50%|█████     | 2/4 [00:04<00:04,  2.31s/it]


  → 2개 완료

[대설] 4개 생성 중...


대설:  50%|█████     | 2/4 [00:04<00:04,  2.35s/it]

  → 2개 완료

===== 총 61개 QA 생성 완료 =====


In [11]:
# [셀6] 생성 결과 확인
from collections import Counter

print(f"총 QA 수: {len(qa_dataset)}개\n")

# 유형별 분포
print("재난 유형별:")
for k, v in Counter(q["disaster_type"] for q in qa_dataset).most_common():
    print(f"  {k}: {v}개")

print("\n질문 유형별:")
for k, v in Counter(q["type"] for q in qa_dataset).most_common():
    print(f"  {k}: {v}개")

# 샘플 5개 출력
print("\n===== 샘플 QA 5개 =====")
for qa in qa_dataset[:5]:
    print(f"\n[{qa['id']}] [{qa['disaster_type']}] [{qa['type']}]")
    print(f"Q: {qa['question']}")
    print(f"A: {qa['gold_answer']}")

총 QA 수: 61개

재난 유형별:
  지진: 12개
  화재: 12개
  황사: 8개
  화학사고재난: 8개
  미세먼지: 6개
  감염병예방: 4개
  산불: 4개
  강풍: 3개
  한파: 2개
  대설: 2개

질문 유형별:
  절차형: 30개
  안전판단형: 24개
  수치법령형: 4개
  복합추론형: 3개

===== 샘플 QA 5개 =====

[Q001] [지진] [절차형]
Q: 집 안에서 지진 발생 시 안전한 대피 공간으로 적합한 장소는 어디인가요?
A: 탁자 아래와 같이 집 안에서 대피할 수 있는 안전한 대피 공간을 미리 파악해 둡니다.

[Q002] [지진] [절차형]
Q: 지진 발생 시, 가족과 만날 장소와 연락 방법을 미리 어떻게 정해두어야 할까요?
A: 가족과 만날 장소와 연락 방법을 미리 정해두고, 위급 상황 시 이를 따라 행동해야 합니다.

[Q003] [지진] [안전판단형]
Q: 지진 발생 시 엘리베이터를 이용하는 것이 안전한지 판단해 보세요.
A: 지진 발생 시 엘리베이터는 멈출 수 있으므로 안전하지 않으며, 계단을 이용하여 건물 밖으로 대피해야 합니다.

[Q004] [지진] [절차형]
Q: 지진 발생 시, 도심 지역에서 안전한 대피 장소로 이동할 때 주의해야 할 사항은 무엇인가요?
A: 도심 지역에서는 깨진 유리창이나 간판 등이 떨어져 다칠 위험이 있으므로, 가까운 공원이나 넓은 공간이 없다면 최근에 지은 튼튼한 건물 안으로 들어가 우선 몸을 보호해야 합니다.

[Q005] [지진] [안전판단형]
Q: 야간 대피 시 안전을 위해 가장 중요한 행동은 무엇인가요?
A: 야간에는 넘어지거나 추락할 위험이 있으므로, 손전등을 사용하여 조심스럽게 대피해야 합니다.


In [12]:
# [셀7] JSON 저장
import json

QA_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(QA_PATH, "w", encoding="utf-8") as f:
    json.dump(qa_dataset, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {QA_PATH}")
print(f"총 {len(qa_dataset)}개 QA")
print("\n다음: 03_run_experiments.ipynb")

저장 완료: D:\충북대\지능화캡스톤\data\qa_dataset.json
총 61개 QA

다음: 03_run_experiments.ipynb


In [13]:
# [추가셀] 수치법령형 + 복합추론형 보충 생성
import json, re, time
from tqdm import tqdm

# 현재 저장된 QA 로드
with open(QA_PATH, "r", encoding="utf-8") as f:
    qa_dataset = json.load(f)

qa_id = len(qa_dataset) + 1

# 유형 강제 지정 프롬프트
SUPPLEMENT_PROMPT = """당신은 재난안전 전문가입니다.
아래 [문서 내용]을 읽고 반드시 [{qtype}] 유형의 QA를 {n}개 만드세요.

[{qtype}] 유형 설명:
{qtype_desc}

규칙:
1. gold_answer는 반드시 [문서 내용]에 있는 구체적인 내용이어야 합니다.
2. 모호하거나 빈 답변("이렇게 합니다" 등)은 절대 금지입니다.
3. JSON 배열만 출력하세요.

[문서 내용]
{context}

[출력 형식]
[
  {{"type": "{qtype}", "question": "질문", "gold_answer": "구체적인 정답"}},
  ...
]"""

TYPE_DESC = {
    "수치법령형": "기준 수치, 농도, 풍속, 온도, 발령 기준 등 구체적인 숫자나 법령 조건을 묻는 질문. 예: '미세먼지 나쁨 단계 기준은?', '태풍 경보 발령 풍속은?'",
    "복합추론형": "두 가지 이상 조건을 결합하거나 특정 대상(노인, 어린이, 운전자 등)의 상황을 가정한 질문. 예: '야간에 홍수 예보 발령 시 지하 주차장에 있다면?', '고령자가 폭염 시 주의해야 할 사항은?'"
}

# 수치가 있을 법한 유형 우선 (지진 제외, 수치 많은 유형 선택)
PRIORITY_TYPES = ["황사", "미세먼지", "화재", "화학사고재난", "강풍", "지진", "감염병예방", "산불"]

def generate_supplement(qtype: str, target: int) -> list:
    collected = []
    for dtype in PRIORITY_TYPES:
        if len(collected) >= target:
            break
        chunks = by_type.get(dtype, [])
        for i in range(0, len(chunks), 5):
            if len(collected) >= target:
                break
            ctx = "\n\n".join(c.page_content for c in chunks[i:i+5])[:2000]
            prompt = SUPPLEMENT_PROMPT.format(
                qtype=qtype,
                qtype_desc=TYPE_DESC[qtype],
                n=2,
                context=ctx
            )
            response = llm.invoke(prompt)
            try:
                cleaned = re.sub(r"```json|```", "", response).strip()
                match = re.search(r"\[.*\]", cleaned, re.DOTALL)
                if match:
                    qas = json.loads(match.group())
                    for qa in qas:
                        if (qa.get("gold_answer") and
                            len(qa["gold_answer"]) > 20 and
                            "이렇게" not in qa["gold_answer"][:10]):
                            collected.append({
                                "id": f"Q{qa_id + len(collected):03d}",
                                "type": qtype,
                                "disaster_type": dtype,
                                "question": qa["question"],
                                "gold_answer": qa["gold_answer"],
                                "source_file": chunks[i].metadata.get("source_file", ""),
                            })
                            if len(collected) >= target:
                                break
            except:
                pass
            time.sleep(0.3)
    return collected

# 수치법령형 10개 추가
print("수치법령형 생성 중...")
new_numeric = generate_supplement("수치법령형", target=10)
print(f"  → {len(new_numeric)}개 생성")

# 복합추론형 10개 추가
print("복합추론형 생성 중...")
new_complex = generate_supplement("복합추론형", target=10)
print(f"  → {len(new_complex)}개 생성")

# 기존 QA에 합치기
qa_dataset.extend(new_numeric)
qa_dataset.extend(new_complex)

# ID 재정렬
for i, qa in enumerate(qa_dataset, 1):
    qa["id"] = f"Q{i:03d}"

# 저장
with open(QA_PATH, "w", encoding="utf-8") as f:
    json.dump(qa_dataset, f, ensure_ascii=False, indent=2)

# 최종 분포 확인
from collections import Counter
print(f"\n최종 QA 수: {len(qa_dataset)}개")
print("\n질문 유형별:")
for k, v in Counter(q["type"] for q in qa_dataset).most_common():
    print(f"  {k}: {v}개")
print("\n재난 유형별:")
for k, v in Counter(q["disaster_type"] for q in qa_dataset).most_common():
    print(f"  {k}: {v}개")

수치법령형 생성 중...
  → 10개 생성
복합추론형 생성 중...
  → 10개 생성

최종 QA 수: 81개

질문 유형별:
  절차형: 30개
  안전판단형: 24개
  수치법령형: 14개
  복합추론형: 13개

재난 유형별:
  황사: 28개
  지진: 12개
  화재: 12개
  화학사고재난: 8개
  미세먼지: 6개
  감염병예방: 4개
  산불: 4개
  강풍: 3개
  한파: 2개
  대설: 2개


In [14]:
# 황사 10개 초과분 제거
from collections import defaultdict
import json

# 유형별로 묶기
by_disaster = defaultdict(list)
for qa in qa_dataset:
    by_disaster[qa["disaster_type"]].append(qa)

# 황사 10개로 제한
by_disaster["황사"] = by_disaster["황사"][:10]

# 다시 합치기 + ID 재정렬
qa_dataset_clean = []
for dtype in ["지진","화재","황사","화학사고재난","미세먼지","감염병예방","산불","강풍","한파","대설"]:
    qa_dataset_clean.extend(by_disaster[dtype])

for i, qa in enumerate(qa_dataset_clean, 1):
    qa["id"] = f"Q{i:03d}"

# 저장
with open(QA_PATH, "w", encoding="utf-8") as f:
    json.dump(qa_dataset_clean, f, ensure_ascii=False, indent=2)

# 결과 확인
print(f"최종 QA 수: {len(qa_dataset_clean)}개\n")
print("재난 유형별:")
for k, v in Counter(q["disaster_type"] for q in qa_dataset_clean).most_common():
    print(f"  {k}: {v}개")
print("\n질문 유형별:")
for k, v in Counter(q["type"] for q in qa_dataset_clean).most_common():
    print(f"  {k}: {v}개")

최종 QA 수: 63개

재난 유형별:
  지진: 12개
  화재: 12개
  황사: 10개
  화학사고재난: 8개
  미세먼지: 6개
  감염병예방: 4개
  산불: 4개
  강풍: 3개
  한파: 2개
  대설: 2개

질문 유형별:
  절차형: 30개
  안전판단형: 24개
  수치법령형: 6개
  복합추론형: 3개


In [15]:
# 황사 제외하고 수치법령형/복합추론형 보충 생성
import json, re, time
from collections import Counter

# 현재 QA 로드
with open(QA_PATH, "r", encoding="utf-8") as f:
    qa_dataset = json.load(f)

qa_id = len(qa_dataset) + 1

# 황사 제외 우선순위 (수치 정보 많은 유형 앞쪽)
PRIORITY_NO_HWANGSA = ["미세먼지", "화학사고재난", "화재", "강풍", "지진", "감염병예방", "산불", "한파", "대설"]

def generate_supplement_v2(qtype: str, target: int) -> list:
    collected = []
    for dtype in PRIORITY_NO_HWANGSA:
        if len(collected) >= target:
            break
        chunks = by_type.get(dtype, [])
        for i in range(0, len(chunks), 5):
            if len(collected) >= target:
                break
            ctx = "\n\n".join(c.page_content for c in chunks[i:i+5])[:2000]
            prompt = SUPPLEMENT_PROMPT.format(
                qtype=qtype,
                qtype_desc=TYPE_DESC[qtype],
                n=3,
                context=ctx
            )
            response = llm.invoke(prompt)
            try:
                cleaned = re.sub(r"```json|```", "", response).strip()
                match = re.search(r"\[.*\]", cleaned, re.DOTALL)
                if match:
                    qas = json.loads(match.group())
                    for qa in qas:
                        if (qa.get("gold_answer") and
                            len(qa["gold_answer"]) > 20 and
                            "이렇게" not in qa["gold_answer"][:10]):
                            collected.append({
                                "id": f"Q{qa_id + len(collected):03d}",
                                "type": qtype,
                                "disaster_type": dtype,
                                "question": qa["question"],
                                "gold_answer": qa["gold_answer"],
                                "source_file": chunks[i].metadata.get("source_file", ""),
                            })
                            if len(collected) >= target:
                                break
            except:
                pass
            time.sleep(0.3)
    return collected

# 수치법령형 10개 추가 목표
print("수치법령형 보충 생성 중... (황사 제외)")
new_numeric = generate_supplement_v2("수치법령형", target=10)
print(f"  → {len(new_numeric)}개 생성")

# 복합추론형 10개 추가 목표
print("복합추론형 보충 생성 중... (황사 제외)")
new_complex = generate_supplement_v2("복합추론형", target=10)
print(f"  → {len(new_complex)}개 생성")

# 합치기 + ID 재정렬
qa_dataset.extend(new_numeric)
qa_dataset.extend(new_complex)
for i, qa in enumerate(qa_dataset, 1):
    qa["id"] = f"Q{i:03d}"

# 저장
with open(QA_PATH, "w", encoding="utf-8") as f:
    json.dump(qa_dataset, f, ensure_ascii=False, indent=2)

print(f"\n최종 QA 수: {len(qa_dataset)}개")
print("\n질문 유형별:")
for k, v in Counter(q["type"] for q in qa_dataset).most_common():
    print(f"  {k}: {v}개")
print("\n재난 유형별:")
for k, v in Counter(q["disaster_type"] for q in qa_dataset).most_common():
    print(f"  {k}: {v}개")

수치법령형 보충 생성 중... (황사 제외)
  → 10개 생성
복합추론형 보충 생성 중... (황사 제외)
  → 10개 생성

최종 QA 수: 83개

질문 유형별:
  절차형: 30개
  안전판단형: 24개
  수치법령형: 16개
  복합추론형: 13개

재난 유형별:
  미세먼지: 21개
  화학사고재난: 13개
  지진: 12개
  화재: 12개
  황사: 10개
  감염병예방: 4개
  산불: 4개
  강풍: 3개
  한파: 2개
  대설: 2개
